In [522]:
using Pkg
Pkg.add("Combinatorics")
Pkg.add("Bits")
Pkg.add("Random")
Pkg.add("Parsers")
Pkg.add("LinearAlgebra")

using LinearAlgebra
using Distributions
using Random
using Bits
using Parsers

   Resolving package versions...
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Project.toml`
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Project.toml`
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Project.toml`
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Project.toml`
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Manifest.toml`
   Resolving package versions...
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Project.toml`
  No Changes to `C:\Users\saika\.julia\environments\v1.8\Manifest.toml`


1) Generate Random starting points. <br>
2) Get individual from starting points. <br>
3) Get starting points from individual. <br>
4) Check if starting points are within acceptable range.

In [590]:
# Random Starting point generator k - length of motif, t is number of sequences
function RandomGenerator(DnaSequenceLength, k, t)
    randomStartPoints = Vector{Int16}(undef, 0)
    for i in range(1, t)
        push!(randomStartPoints, ceil(Int, rand()*(DnaSequenceLength - k + 1)))
    end
    return randomStartPoints
end

# represent each starting point as 16 bits and concat them together and return
function getIndividual(randomStartingPoints)
    result = ""
    for i in randomStartingPoints
        temp = string(bits(Int16(i)))
        temp = replace(temp, "<" => "")
        temp = replace(temp, ">" => "")
        temp =replace(temp, " " => "")
        result *= temp
    end
    return result
end

# get numericIndividual from the individual
function getNumericIndividual(regularIndividual)
    result = []
    numIndividuals = Int(length(regularIndividual)/16)
    for i in range(0, numIndividuals-1)
        temp = regularIndividual[Int(16*i+1):Int(16*i+16)]
        push!(result, parse(Int64, temp; base=2))
    end
    return result
    
end


# check if starting points are valid in numeric Individual
function validateIndividual(regularIndividual, DnaSequences, k)
    numericIndividual = getNumericIndividual(regularIndividual)
    for i in range(1, length(numericIndividual))
        if numericIndividual[i] > length(DnaSequences[1]) - k + 1
            return false
        end
    end
    return true
end

validateIndividual (generic function with 1 method)

1) Get Fitness score of a set of motifs for a set of starting points.

In [524]:
# pseudo count for each nucelotide is 1 and sum of pseudo counts = 4(1) = 4
function getFitnessScore(motifs, k, t)
    ScorerDict = Dict([
        ("A", zeros(k)),
        ("C", zeros(k)),
        ("G", zeros(k)),
        ("T", zeros(k))
    ])
    #println(motifs)
    S = 0
    # A, C, G, T order 
    bgCount = Dict([
        ("A", 0),
        ("C", 0),
        ("G", 0),
        ("T", 0)            
    ])
    for col in range(1, k)
        for row in range(1, t)
            ScorerDict[string(motifs[row][col])][col] += 1
            S += 1
            bgCount[string(motifs[row][col])] += 1
        end
    end

    fitness = 0
    for col in range(1, k)
        informationContent = 0
        for nuc in ["A", "C", "T", "G"]
            freqObs = (ScorerDict[nuc][col] + 1)/(t - 1 + 4)
            freqBg = (bgCount[nuc] + 1)/(S + 4)
            informationContent += freqObs*(log(2, (freqObs/freqBg)))
        end
        fitness += informationContent
    end

    return fitness
end


getFitnessScore (generic function with 1 method)

1) perform crossover. <br>
2) perform mutation. <br>
3) perform selection - returns indices of selected individuals - roulette wheel selection based on their fitness scores

In [525]:
# get 2 bit strings and output 2 offspring bit strings
function performCrossover(parent1, parent2)
    
    crossPosition = Int(ceil(rand()*length(parent1)))
    #println(crossPosition)
    
    offspring1 = parent1[1:crossPosition]*parent2[crossPosition+1:end]
    offspring2 = parent1[crossPosition+1:end]*parent2[1:crossPosition]
    
    return offspring1, offspring2
end

# get 1 bit string and output 1 mutated bit string
function performMutation(regularIndividual, mutationRate, mutationProb)
    
    for i in range(1, mutationRate)
        mutPosition = Int(ceil(rand()*length(regularIndividual)))
        #println(mutPosition)
        currProb = rand()
        #println(currProb)
        temp = '0'
        if(currProb <= mutationProb)
            # perform mutation on that random bit
            if(regularIndividual[mutPosition] == '1')
                temp = '0'
            else
                temp = '1'
            end
            if mutPosition == 1
                return temp*regularIndividual[2:end]
            else
                return regularIndividual[1:mutPosition-1]*temp*regularIndividual[mutPosition+1:end]
            end
        end 
    end
end

# roulette wheel selection operation 
function performSelection(numSelections, fitnessScores)
    
    normalizedFitnessScores = normalize(fitnessScores)
    individualIndices = []
    for i in range(1, length(fitnessScores))
        push!(individualIndices, i)
    end
    selectedIndices = wsample(individualIndices, normalizedFitnessScores, numSelections, replace = false)
    
    return selectedIndices
end

performSelection (generic function with 1 method)

In [526]:
performSelection(2, [0.25, 0.25, 0.25, 0.25])

2-element Vector{Any}:
 3
 4

In [527]:
# using shift width to evaluate all "possible" candidates in proximity
# motif width k and number of sequences t 
function evaluateCandidate(DnaSequences, numericIndividual, shiftWidth, k, t)
    maxFitness = 0
    bestWidth = 0
    motifs = []
    #println(numericIndividual) 
    #println(DnaSequences)
    for i in range(-shiftWidth, shiftWidth)
        #print(i)
        # get motifs based on starting positions in individual
        for j in range(1, t)
            if numericIndividual[j] + i < 1 || numericIndividual[j] + i > length(DnaSequences[1]) - k + 1
                motifs = []
                break
            end
            push!(motifs, DnaSequences[j][numericIndividual[j] + i:numericIndividual[j] + i + k - 1])
        end
        if length(motifs) != 0 
            currFitness = getFitnessScore(motifs, k, t)
            #println(i, currFitness, motifs)
            if(currFitness > maxFitness)
                maxFitness = currFitness
                bestWidth = i
            end
            motifs = []
        end
    end
    
    # return the updatedIndividual and its fitness score
    updatedIndividual = []
    for i in range(1, t)
        push!(updatedIndividual, numericIndividual[i] + bestWidth)
    end
    return updatedIndividual, maxFitness
end

evaluateCandidate (generic function with 1 method)

1) Code to generate DNA sequences

In [528]:
function AddMutation(m)
    Nuc = ["A", "G", "T", "C"]
    Nmutations = 0
    mutated = m
    # introducing mutations in the motif 
    for j in range(1, Nmutations)
        randomIndex = Int(ceil(rand()*length(m)))
        if randomIndex == 1
            mutated = Nuc[Int(ceil(rand()*4))] * m[2:length(m)]
        elseif randomIndex == length(m)
            mutated = m[1:(length(m)-1)] * Nuc[Int(ceil(rand()*4))]
        else
            mutated = m[1:randomIndex-1] * Nuc[Int(ceil(rand()*4))] * m[randomIndex+1:length(m)]
        end
    end    
    
    return mutated
end


function DnaGenerator(t, m, l)
    DnaSequences = []
    GC = ["G", "C"]
    AT = ["A", "T"]
    lengthExceptMotif = l - length(m)
    baseSequence = ""
    # Generating Dna Sequences, apart from the motif, with 41% GC content
    for i in range(1, ceil(0.41 * lengthExceptMotif))
        baseSequence *= GC[ceil(Int, rand()*2)]
    end
    # Generating Dna Sequences, apart from the motif, with remaining % of AT content
    for i in range(1, lengthExceptMotif - ceil(0.41 * lengthExceptMotif))
        baseSequence *= AT[ceil(Int, rand()*2)]
    end
    
    mutatedMotifs = []
    
    # introducing a different mutation in each motif to be implanted.
    for i in range(1, t)
        push!(mutatedMotifs, AddMutation(m))
    end
    
    #println(mutatedMotifs)
    # adding the motif to each randomly generated sequence
    for i in range(1, t)
        permSequence = join(collect(baseSequence)[randperm(length(baseSequence))])

        insertAt = ceil(Int, rand()*(lengthExceptMotif+1))

        if insertAt == 1
            endSequence = mutatedMotifs[i] * permSequence
        elseif insertAt == lengthExceptMotif + 1
            endSequence = permSequence * mutatedMotifs[i]
        else
            endSequence = permSequence[1:insertAt-1] * mutatedMotifs[i] * permSequence[insertAt:lengthExceptMotif]
        end
        push!(DnaSequences, endSequence)
    end
    return DnaSequences
end


DnaGenerator (generic function with 1 method)

In [529]:
DnaSequences = DnaGenerator(2, "ACTG", 8)

2-element Vector{Any}:
 "GTTACTGG"
 "TGGACTGT"

In [530]:
numericIndividual = RandomGenerator(8, 4, 2)
individual = getIndividual(Int16[2, 2])
evaluateCandidate(DnaSequences, Int16[3, 4], 1, 4, 2)

(Any[2, 3], 1.6586551462714367)

In [599]:
function Execute_MDGA()
    # set motif width 
    k = 8
    # set sequence length 
    l = 20
    # set number of sequences 
    t = 10
    # set crossover probability 
    crossProb = 1
    # set mutation rate
    mutationRate = 1
    # set mutation probability
    mutationProb = 1
    # set shiftWidth 
    shiftWidth = 1
    # repeat experiment g times
    g = 100
    # set population size 
    popSize = 100
    
    
    # generate DNA sequences with the motif implanted in them at random positions
    DnaSequences = DnaGenerator(t, "AAAAAAAA", l)
    println(DnaSequences)
    # generate first population of individuals - 4 individuals(Test)
    numericIndividuals = []
    regularIndividuals = []

    for i in range(1, popSize)
        # generate a set of random starting points
        numericIndividual = RandomGenerator(l, k, t)
        push!(numericIndividuals, numericIndividual)
        # get the corresponding individual 
        regularIndividual = getIndividual(numericIndividual)
        push!(regularIndividuals, regularIndividual)
    end
    
    #println(numericIndividuals)
    #println(regularIndividuals)
    # evaluate current individuals in population - with shift width and get their best scores
    fitnessScores = []
    for i in range(1, popSize)
        updatedIndividual, fitnessScore = evaluateCandidate(DnaSequences, numericIndividuals[i], shiftWidth, k, t)
        #println(updatedIndividual)
        #println(fitnessScore)
        push!(fitnessScores, fitnessScore)
        numericIndividuals[i] = updatedIndividual
        regularIndividuals[i] = getIndividual(updatedIndividual)
    end
    
    #println(fitnessScores)
    
    # iterations begin    
    for i in range(1, g)
        print("iteration ")
        println(i)
        # select 2 parents from the population 
        selectedIndices = performSelection(2, fitnessScores)
        #println(selectedIndices)
        childCount = 0
        # for each pair perform crossover and mutation
        for j in range(1, length(selectedIndices), step=2)
            #print("j ")
            #println(j)
            parent1 = regularIndividuals[selectedIndices[Int(j)]]
            parent2 = regularIndividuals[selectedIndices[Int(j)+1]]
            # perform crossover
            offspring1, offspring2 = performCrossover(parent1, parent2)
            # perform mutation
            mutatedIndividual1 = performMutation(offspring1, mutationRate, mutationProb)
            mutatedIndividual2 = performMutation(offspring2, mutationRate, mutationProb)  
            
            if validateIndividual(mutatedIndividual1, DnaSequences, k)
                mutatedNumericIndividual1 = getNumericIndividual(mutatedIndividual1)
                # evaluate child1
                updatedMutatedNumericIndividual1, fitnessScore = evaluateCandidate(DnaSequences, mutatedNumericIndividual1, shiftWidth, k, t)
                push!(regularIndividuals, mutatedIndividual1)
                push!(fitnessScores, fitnessScore)
                push!(numericIndividuals, updatedMutatedNumericIndividual1)
                childCount += 1
            end
            if validateIndividual(mutatedIndividual2, DnaSequences, k)
                mutatedNumericIndividual2 = getNumericIndividual(mutatedIndividual2)
                # evaluate child2
                updatedMutatedNumericIndividual2, fitnessScore = evaluateCandidate(DnaSequences, mutatedNumericIndividual2, shiftWidth, k, t)
                push!(regularIndividuals, mutatedIndividual2)
                push!(fitnessScores, fitnessScore)
                push!(numericIndividuals, updatedMutatedNumericIndividual2)
                childCount += 1
            end
        end

        
        #println(childCount)

        # remove the worst performing ones
        sortHelper = sortperm(fitnessScores, rev = true)
        
        fitnessScores = fitnessScores[sortHelper]
        numericIndividuals = numericIndividuals[sortHelper]
        regularIndividuals = regularIndividuals[sortHelper]
        #println(regularIndividuals)
        # change 4 to number of individuals in population
        fitnessScores = fitnessScores[1:popSize]
        numericIndividuals = numericIndividuals[1:popSize]
        regularIndividuals = regularIndividuals[1:popSize]
    end
    
    return numericIndividuals[1], fitnessScores[1]
            
end

Execute_MDGA (generic function with 1 method)

In [600]:
Execute_MDGA()

Any["AAAAAAAATAGGGGTTTCAA", "GTAAAAAAAAGGCAGTTATA", "GGGAAAAAAAAAATGTTTCA", "TAGTTAAAAAAAATAAGGGC", "GCAGTTAAAAAAAAGAGTTA", "TCGATAAAAAAAAAAGTTGG", "TGTGAAAAAAAAATAGATCG", "TTGACAAAAAAAAGTAGTGA", "GGTCTAAAAAAAAATATAGG", "AAAAAAAAATGTATGACTGG"]
iteration 1
iteration 2
iteration 3
iteration 4
iteration 5
iteration 6
iteration 7
iteration 8
iteration 9
iteration 10
iteration 11
iteration 12
iteration 13
iteration 14
iteration 15
iteration 16
iteration 17
iteration 18
iteration 19
iteration 20
iteration 21
iteration 22
iteration 23
iteration 24
iteration 25
iteration 26
iteration 27
iteration 28
iteration 29
iteration 30
iteration 31
iteration 32
iteration 33
iteration 34
iteration 35
iteration 36
iteration 37
iteration 38
iteration 39
iteration 40
iteration 41
iteration 42
iteration 43
iteration 44
iteration 45
iteration 46
iteration 47
iteration 48
iteration 49
iteration 50
iteration 51
iteration 52
iteration 53
iteration 54
iteration 55
iteration 56
iteration 57
iteration 58
iteration 5

(Any[3, 3, 11, 6, 9, 6, 11, 8, 12, 5], 2.9373101389506306)

In [ ]:
[10, 12, 7, 13, 7, 13, 10, 9, 8, 11]